# False Belief Test — Methodology & Scoring
Evaluates GPT-2 checkpoints on Trott et al. (2023) stimuli using log-probability scoring.

In [ ]:
import sys, json
from pathlib import Path
import pandas as pd
import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))
from eval_fbt import location_log_prob, evaluate

DATA_PATH = ROOT / 'data' / 'fbt' / 'fb.csv'
df = pd.read_csv(DATA_PATH)
print(f'{len(df)} total items')

## Stimulus structure

In [ ]:
# Filter to standard mention-order subset (first=Start, recent=End)
df_filtered = df[(df['first_mention'] == 'Start') & (df['recent_mention'] == 'End')]
# Drop rows with missing or empty passage/location values
df_filtered = df_filtered.dropna(subset=['passage', 'start', 'end'])
df_filtered = df_filtered[
    df_filtered['passage'].str.strip().ne('') &
    df_filtered['start'].str.strip().ne('') &
    df_filtered['end'].str.strip().ne('')
].reset_index(drop=True)
print(f'{len(df_filtered)} items after mention-order filter and cleaning')
df_filtered[['condition','knowledge_cue','passage','start','end']].head(4)

## Scoring: log P(location | context)

In [ ]:
# Show one scored example
sample = df_filtered.iloc[0]
context = sample['passage'].replace('[MASK]', '').rstrip()

print('Passage:', sample['passage'])
print('Condition:', sample['condition'])
print('Cue type:', sample['knowledge_cue'])
print('Start location:', sample['start'])
print('End location:', sample['end'])

## Score all conditions — final checkpoint

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

CONDITIONS = ['flat', 'chunked', 'balanced']
LABELS = {'flat': 'Sliding-Window', 'chunked': 'EOS-Chunked', 'balanced': 'DENSITY-Balanced'}

def get_final_checkpoint(condition):
    models_dir = ROOT / 'models' / condition
    ckpts = sorted(
        [d for d in models_dir.iterdir() if d.is_dir() and d.name.startswith('checkpoint-')],
        key=lambda d: int(d.name.split('-')[1])
    )
    return ckpts[-1] if ckpts else None

results_summary = {}
for cond in CONDITIONS:
    ckpt = get_final_checkpoint(cond)
    if ckpt is None:
        print(f'{cond}: no checkpoint found')
        continue
    print(f'Loading {cond}: {ckpt.name}')
    tokenizer = GPT2TokenizerFast.from_pretrained(str(ckpt))
    model = GPT2LMHeadModel.from_pretrained(str(ckpt)).to(device)
    model.eval()
    r = evaluate(model, tokenizer, df_filtered, device)
    results_summary[cond] = r
    print(f'  Overall: {r["overall_accuracy"]:.3f}  FB: {r["by_condition"].get("False Belief", float("nan")):.3f}  TB: {r["by_condition"].get("True Belief", float("nan")):.3f}')
    del model

## Summary table

In [ ]:
rows = []
for cond, r in results_summary.items():
    rows.append({
        'Condition': LABELS[cond],
        'Overall': round(r['overall_accuracy'], 3),
        'False Belief': round(r['by_condition'].get('False Belief', float('nan')), 3),
        'True Belief':  round(r['by_condition'].get('True Belief',  float('nan')), 3),
        'Explicit FB':  round(r['by_condition_x_cue'].get('False Belief × Explicit', float('nan')), 3),
        'Implicit FB':  round(r['by_condition_x_cue'].get('False Belief × Implicit', float('nan')), 3),
    })
pd.DataFrame(rows).set_index('Condition')

## Item-level error analysis

In [ ]:
# Show incorrectly answered False Belief items for flat model
if 'flat' in results_summary:
    errors = pd.DataFrame([
        i for i in results_summary['flat']['items']
        if not i['correct'] and 'false' in i['condition'].lower()
    ])
    print(f'{len(errors)} False Belief errors (flat model)')
    if len(errors):
        errors[['knowledge_cue','start_location','end_location','model_choice','lp_start','lp_end']].head(6)